# Kaggle Qwen2.5 3B GRPO Run

Use this notebook from the Kaggle-backed VS Code kernel. It runs the complementary 3B Unsloth GRPO experiment while the campus server runs the default 1.5B job.

## 1. GPU Check

Expected on Kaggle: two T4 GPUs. If this cell does not show T4s, the notebook is not attached to the Kaggle kernel.

In [1]:
!nvidia-smi

Sat Apr 25 03:59:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Kaggle Working Directory And Repo

The Kaggle kernel cannot see your local laptop path. This section resets the remote kernel to `/kaggle/working`, then clones or updates the pushed `round2-redshift` branch. Run these cells even if a previous `%cd` failed.


In [40]:
import os
from pathlib import Path

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")
print("cwd:", os.getcwd())


cwd: /kaggle/working


In [41]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "round2-redshift"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f"{WORKDIR.name}.bak.{int(time.time())}")
        shutil.move(str(WORKDIR), str(backup))
        print("Moved non-git existing directory to", backup)
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            round2-redshift -> FETCH_HEAD
   091fc35..5ec7173  round2-redshift -> origin/round2-redshift
Already on 'round2-redshift'


Your branch is behind 'origin/round2-redshift' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 091fc35..5ec7173
Fast-forward
 scripts/run_kaggle_qwen3b_grpo.sh |  6 +++--
 scripts/train_unsloth_grpo.py     | 55 ++++++++++++++++++++++++++++++++++++---
 2 files changed, 56 insertions(+), 5 deletions(-)
cwd: /kaggle/working/MetaHackathon-R2
5ec7173 fix: keep unsloth grpo lora trainable
091fc35 fix: make kaggle notebook bootstrap repo safely
bee18a1 fix: make kaggle grpo runner import package
38e9283 training: add kaggle qwen3b grpo runner
3f54d9d docs: add unsloth grpo runbook


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            round2-redshift -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [42]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
!python - <<'PY'
import os
from pathlib import Path
print("cwd=", Path.cwd())
print("PYTHONPATH=", os.environ.get("PYTHONPATH"))
print("repo exists=", Path("src/oncallenv").exists())



/kaggle/working/MetaHackathon-R2
env: PYTHONPATH=src:scripts
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
cwd= /kaggle/working/MetaHackathon-R2
PYTHONPATH= src:scripts
repo exists= True


## 3. Install Dependencies

Run the normal install first. If Kaggle has a PyTorch/CUDA dependency conflict, use the fallback cell after it.

In [43]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt

In [44]:
# Fallback install if the previous cell fails:
# !python -m pip install -U transformers datasets accelerate trl peft bitsandbytes unsloth

## 4. Verify Environment

Expected: `21 passed` and OpenEnv validation OK.

In [45]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

.....................                                                    [100%]
21 passed in 4.91s
[OK] : Ready for multi-mode deployment


## 5. Small 3B Smoke Run

This is intentionally small. Continue only if it writes `training_results/unsloth_grpo_qwen3b_smoke/summary.json`.

In [46]:
!bash scripts/run_kaggle_qwen3b_grpo.sh smoke

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 00:45:53.953071: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777077953.976606     980 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777077953.984531     980 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777077954.004050     980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777077954.004082     980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [47]:
!cat training_results/unsloth_grpo_qwen3b_smoke/summary.json

{
  "model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
  "duration_sec": 1472.0738599300385,
  "max_steps": 50,
  "num_train_tasks": 40,
  "num_eval_tasks": 8,
  "curriculum_buffer": "curriculum_results/buffer.json",
  "baseline_mean_reward": 0.261484375,
  "trained_mean_reward": 0.31451562499999997,
  "adapter_dir": "training_results/unsloth_grpo_qwen3b_smoke/adapter",
  "dataset_path": "training_results/unsloth_grpo_qwen3b_smoke/dataset.jsonl",
  "trainable_parameter_report": {
    "total": 1728606208,
    "trainable": 29933568,
    "lora_tensors": 504
  }
}

## 6. Main Kaggle 3B Run

This is the main complementary run: Qwen2.5 3B, 160 curriculum tasks, 600 GRPO steps.

In [49]:
!bash scripts/run_kaggle_qwen3b_grpo.sh main

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 01:17:23.241243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777079843.264508    1261 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777079843.272565    1261 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777079843.293539    1261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777079843.293584    1261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_kaggle/summary.json
!bash scripts/run_kaggle_qwen3b_grpo.sh summary

## 7. Optional Extended Run

Run this only if the main 600-step job is stable and Kaggle time remains.

In [ ]:
# !bash scripts/run_kaggle_qwen3b_grpo.sh long

## 8. OOM Fallback

If the 3B smoke run OOMs, run this 1.5B ablation instead. It still gives a Kaggle portability result.

In [ ]:
# !bash scripts/run_kaggle_qwen3b_grpo.sh fallback-1b5

## 9. Archive Outputs

Download `/kaggle/working/qwen3b_grpo_results.tar.gz` from Kaggle outputs.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh archive